In [0]:
from pyspark.sql.functions import current_timestamp, lit
import datetime

# Read existing bronze table as base data
df_base = spark.read.table("fraud_analytics.bronze.credit_card_fraud")

# Simulate 3 daily files arriving in ADLS2 bronze container
storage_account_name = "shruthistorageaccount"
bronze_path = f"abfss://bronze@{storage_account_name}.dfs.core.windows.net/"

# Write 3 simulated daily files
df_base.limit(1000) \
    .write.format("csv") \
    .option("header", "true") \
    .mode("overwrite") \
    .save(bronze_path + "incoming/transactions_2026_03_16.csv")

df_base.limit(800) \
    .write.format("csv") \
    .option("header", "true") \
    .mode("overwrite") \
    .save(bronze_path + "incoming/transactions_2026_03_17.csv")

df_base.limit(1200) \
    .write.format("csv") \
    .option("header", "true") \
    .mode("overwrite") \
    .save(bronze_path + "incoming/transactions_2026_03_18.csv")

print("3 daily files created in bronze/incoming/ folder!")

In [0]:
# Auto Loader configuration
checkpoint_path = f"abfss://bronze@{storage_account_name}.dfs.core.windows.net/checkpoints/autoloader"

# Read ALL files in incoming/ folder using Auto Loader
df_autoloader = spark.readStream \
    .format("cloudFiles") \
    .option("cloudFiles.format", "csv") \
    .option("cloudFiles.schemaLocation", checkpoint_path + "/schema") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(bronze_path + "incoming/")

# Write stream to bronze Delta table
query = df_autoloader.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", checkpoint_path) \
    .option("mergeSchema", "true") \
    .trigger(availableNow=True) \
    .toTable("fraud_analytics.bronze.daily_transactions")

# Wait for stream to finish
query.awaitTermination()

print("Auto Loader completed!")

In [0]:
# Simulate a NEW file arriving today
df_base.limit(500) \
    .write.format("csv") \
    .option("header", "true") \
    .mode("overwrite") \
    .save(bronze_path + "incoming/transactions_2026_03_19.csv")

print("New file added!")

In [0]:
# Verify Auto Loader results
spark.sql("""
    SELECT 
        COUNT(*) as total_rows
    FROM fraud_analytics.bronze.daily_transactions
""").show()

# Check checkpoint was created
dbutils.fs.ls(checkpoint_path)